# Cap rates by cross-prediction

Score for-sale listings with the *rental* model of the same property type to estimate the rent
they could achieve, then turn that into a yield:

```
gross_cap_rate = estimated_monthly_rent * 12 / asking_price
```

This works because feature availability follows `property_type`, not `listing_type` -- a sale
frame and a rent frame of the same property type have identical model columns.

**Gross**, not net: it ignores vacancy, management, maintenance and tax. Use it to rank, not to
underwrite.

Prerequisite: both models saved by `01_train_segment.ipynb`.

In [1]:
PROPERTY_TYPE = "apartment"  # apartment | house
MIN_LISTINGS_PER_DISTRICT = 10

In [2]:
import plotly.express as px

from inmoai_lt.modeling import SegmentModel, cap_rate_by_district, estimate_cap_rate, load_segment

sale_segment = f"{PROPERTY_TYPE}_sale"
rent_segment = f"{PROPERTY_TYPE}_rent"

sale_model = SegmentModel.load(sale_segment)
rent_model = SegmentModel.load(rent_segment)
sale_listings = load_segment(sale_segment)

print(f"{len(sale_listings):,} {sale_segment} listings")
print(f"rent model trained on {rent_model.metadata['n_train']:,} rows, low_n={rent_model.low_n}")

3,036 apartment_sale listings
rent model trained on 1,303 rows, low_n=False


## 1. Estimate rent and cap rate per listing

Passing `sale_model` as well adds its own price estimate, so a listing can be read on two axes:
yield (`gross_cap_rate_pct`) and whether it is asking above or below model value
(`price_premium_pct`).

In [3]:
cap_rates = estimate_cap_rate(sale_listings, rent_model, sale_model=sale_model)
cap_rates.head(10)

,listing_id,district,total_area_sqm,asking_price_eur,estimated_monthly_rent_eur,estimated_annual_rent_eur,gross_cap_rate_pct,low_confidence,predicted_price_eur,price_premium_pct
0,1-2796291,Pašilaičiai,64.00,180000.0,701.0,8412.0,4.67,False,193186.0,-6.83
1,1-2999625,Pilaitė,70.54,168000.0,643.0,7720.0,4.60,False,177980.0,-5.61
2,1-3010389,Pilaitė,75.00,265000.0,799.0,9592.0,3.62,False,254736.0,4.03
3,1-3057747,Pilaitė,51.00,218000.0,758.0,9100.0,4.17,False,207883.0,4.87
4,1-3094223,Markučiai,40.00,59900.0,779.0,9351.0,15.61,False,45962.0,30.33
5,1-3098293,Naujamiestis,34.89,149990.0,610.0,7323.0,4.88,False,145117.0,3.36
6,1-3186466,Senamiestis,60.00,337000.0,870.0,10444.0,3.10,False,354219.0,-4.86
7,1-3190522,Pilaitė,30.67,129000.0,529.0,6348.0,4.92,False,141273.0,-8.69
8,1-3222500,Šnipiškės,28.59,50000.0,463.0,5559.0,11.12,False,39367.0,27.01
9,1-3234516,Šnipiškės,19.00,69000.0,429.0,5152.0,7.47,False,63135.0,9.29


In [4]:
# low_confidence = the rent model is low-n, or the listing's district was never seen in rent
# training (target encoding falls back to the global mean, so the estimate has no local signal).
flagged = int(cap_rates["low_confidence"].sum())
print(f"low_confidence: {flagged:,} of {len(cap_rates):,} listings")
cap_rates["gross_cap_rate_pct"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).round(2)

low_confidence: 11 of 3,036 listings


count    3036.00
mean        5.20
std         2.46
min         0.32
5%          2.91
25%         4.11
50%         4.79
75%         5.67
95%         8.52
max        50.12
Name: gross_cap_rate_pct, dtype: float64

In [5]:
fig = px.histogram(
    cap_rates[~cap_rates["low_confidence"]],
    x="gross_cap_rate_pct",
    nbins=80,
    range_x=[0, 15],
    title=f"{PROPERTY_TYPE}: gross cap rate distribution (low-confidence rows excluded)",
    labels={"gross_cap_rate_pct": "gross cap rate (%)"},
)
fig.show()

## 2. Which districts yield most

Medians, and only for districts with enough listings to mean anything.

In [6]:
by_district = cap_rate_by_district(cap_rates, min_listings=MIN_LISTINGS_PER_DISTRICT)
by_district

,listings,median_cap_rate_pct,median_price_eur,median_monthly_rent_eur
district,,,,
Grigiškės,25,6.74,98000.0,616.0
Naujoji Vilnia,76,6.68,105000.0,542.0
Naujininkai,88,6.60,124000.0,676.0
Žemieji Paneriai,18,5.56,127991.0,624.0
Visoriai,24,5.11,218000.0,888.0
Burbiškės,19,5.08,274900.0,1284.0
Viršuliškės,64,5.03,156500.0,704.0
Baltupiai,88,5.03,184450.0,752.5
Žirmūnai,198,4.97,179999.5,782.5


In [7]:
fig = px.bar(
    by_district.reset_index(),
    x="median_cap_rate_pct",
    y="district",
    orientation="h",
    hover_data=["listings", "median_price_eur", "median_monthly_rent_eur"],
    title=f"{PROPERTY_TYPE}: median gross cap rate by district",
    labels={"median_cap_rate_pct": "median gross cap rate (%)"},
    height=max(400, 22 * len(by_district)),
)
fig.update_yaxes(categoryorder="total ascending")
fig.show()

## 3. Shortlist

High estimated yield *and* asking below what the sale model predicts. Two independent models
have to agree, which filters out listings that only look cheap because they are poor stock.

Note the failure mode this cannot see: a listing may be cheap for a reason absent from the
features (legal issues, a bad neighbour, a photo the model never looked at).

In [8]:
shortlist = cap_rates[
    ~cap_rates["low_confidence"] & (cap_rates["price_premium_pct"] < 0)
].sort_values("gross_cap_rate_pct", ascending=False)

shortlist.head(20)

,listing_id,district,total_area_sqm,asking_price_eur,estimated_monthly_rent_eur,estimated_annual_rent_eur,gross_cap_rate_pct,low_confidence,predicted_price_eur,price_premium_pct
174,1-3622570,Naujamiestis,23.12,12466.0,521.0,6248.0,50.12,False,72558.0,-82.82
81,1-3553490,Pavilnys,35.28,19159.0,544.0,6525.0,34.06,False,21160.0,-9.46
123,1-3595970,Naujoji Vilnia,52.38,27119.0,698.0,8370.0,30.87,False,31123.0,-12.87
80,1-3553484,Rasos,41.64,37690.0,843.0,10110.0,26.82,False,48403.0,-22.13
95,1-3574861,Rasos,36.06,30473.0,665.0,7985.0,26.20,False,37148.0,-17.97
233,1-3638557,Naujininkai,33.04,35000.0,731.0,8767.0,25.05,False,110799.0,-68.41
992,1-3674572,Naujininkai,50.19,45000.0,848.0,10182.0,22.63,False,48844.0,-7.87
1962,1-3686765,Salininkai,62.07,45000.0,809.0,9706.0,21.57,False,100128.0,-55.06
365,1-3649259,Verkiai,50.81,49995.0,891.0,10689.0,21.38,False,66672.0,-25.01
351,1-3648445,Užupis,54.00,47000.0,790.0,9483.0,20.18,False,208648.0,-77.47
